# Structured Outputs in LangChain

This notebook demonstrates how to direct LLMs to return structured outputs (like Pydantic models, TypedDicts, or Dataclasses) instead of raw text. Generating structured outputs is vital for downstream processing in software pipelines.

## Step 1: Environment Setup
We load API keys from the `.env` file and set the required environment variables.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


## Step 2: Initialize Chat Model
We initialize a chat model using LangChain's unified `init_chat_model` utility.

### Alternative Models & Providers
You can switch between the following popular models by changing the `model` and `model_provider` parameters:

* **Groq Models (`model_provider="groq"`):**
  * `llama-3.3-70b-versatile` (Large, highly capable model)
  * `llama-3.1-8b-instant` (Fast, lightweight model)
  * `mixtral-8x7b-32768` (Mixture of Experts model)
  * `gemma2-9b-it` (Google Gemma model hosted on Groq)

* **Google Gemini Models (`model_provider="google_genai"`):**
  * `gemini-2.0-flash` (Next-gen high performance multimodal model)
  * `gemini-1.5-flash` (Standard fast multimodal model)
  * `gemini-1.5-pro` (Highly advanced reasoning model)

In [2]:
from langchain.chat_models import init_chat_model

# Option A: Groq (using Llama 3)
model = init_chat_model("llama-3.1-8b-instant", model_provider="groq")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000022F6D1C9010>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000022F6D1C9A90>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

## Step 3: Define a Target Schema (Pydantic Model)
We use Pydantic to describe the properties we want the LLM to extract.

In [35]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie.")
    year:int=Field(description="This year the movie was released.")
    director:str=Field(description="The director of the movie.")
    Actors:str=Field(description="Lead actors like hero and heroine.")
    ratings:float=Field(description="MOvies ratings out of 10.")
    Plot:str=Field(description="The genre of the movie")


## Step 4: Output Comparison

### Case A: Standard Unstructured Output
Calling the model directly returns a standard `AIMessage` with free-form text.

In [36]:
# Without structured output
model.invoke("Provide the details of the movie IT")

AIMessage(content='IT is a 2017 American supernatural horror film directed by Andy Muschietti, based on the 1986 novel of the same name by Stephen King. The film is the first installment of a planned two-part adaptation.\n\n**Plot:**\n\nThe story revolves around a group of young friends in Derry, Maine, who call themselves "The Losers Club." They consist of:\n\n1. Bill Denbrough (Jaeden Lieberher), the leader of the group who has a stutter and lost his younger brother to the entity.\n2. Beverly Marsh (Sophia Lillis), the brave and strong-willed girl who has a difficult home life.\n3. Ben Hanscom (Jeremy Ray Taylor), the overweight and gentle giant who is also an artist.\n4. Richie Tozier (Finn Wolfhard), the comic relief and a wise-cracking kid.\n5. Mike Hanlon (Chosen Jacobs), the only African American member of the group and the historian of the group.\n6. Stan Uris (Wyatt Oleff), the son of a Jewish family and a skeptic.\n7. Eddie Kaspbrak (Jack Dylan Grazer), the hypochondriac and 

### Case B: Structured Output Binding
Using `.with_structured_output(Movie)` wraps the model and validates inputs to guarantee a structured Pydantic object.

In [37]:
# With structured output.
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000022F6D1C9010>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000022F6D1C9A90>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'pro

In [40]:
model_with_structure.invoke("Provide the details of the movie The Amazing Spider")

Movie(title='The Amazing Spider', year=2012, director='Marc Webb', Actors='Andrew Garfield, Emma Stone, Rhys Ifans', ratings=7.7, Plot='Action, Adventure, Sci-Fi')

## Step 5: Returning Raw Messages alongside Parsed Data
Using `include_raw=True` yields a dictionary containing the original `AIMessage` (`raw`), the parsed Pydantic object (`parsed`), and any error information (`parsing_error`).

In [41]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(...,description="The title of the movie.")
    year:int=Field(...,description="This year the movie was released.")
    director:str=Field(...,description="The director of the movie.")
    Actors:str=Field(...,description="Lead actors like hero and heroine.")
    ratings:float=Field(...,description="MOvies ratings out of 10.")
    Plot:str=Field(...,description="The genre of the movie")

model_with_parsed_strct=model.with_structured_output(Movie,include_raw=True)
model_with_parsed_strct

{
  raw: _ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000022F6D1C9010>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000022F6D1C9A90>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameter

In [58]:
model_with_parsed_strct.invoke("Provide the details of the movie Inception")

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'xgfa2taj0', 'function': {'arguments': '{"Actors":"Leonardo DiCaprio, Joseph Gordon-Levitt, Marion Cotillard, Ellen Page, Tom Hardy, Ken Watanabe, Cillian Murphy","Plot":"Action, Thriller","director":"Christopher Nolan","ratings":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 316, 'total_tokens': 394, 'completion_time': 0.114448268, 'completion_tokens_details': None, 'prompt_time': 0.01871882, 'prompt_tokens_details': None, 'queue_time': 0.05341121, 'total_time': 0.133167088}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f0d60-3e7a-7e22-91d5-ef3db7c596ac-0', tool_calls=[{'name': 'Movie', 'args': {'Actors': 'Leonardo DiCaprio, Joseph Gordon-Levitt, Marion Coti

## Step 6: Nested Data Structures
Schemas can be nested (e.g. containing lists of other Pydantic model objects) to extract complex hierarchical entities.

In [59]:
from pydantic import BaseModel,Field 
class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None =Field(None,description="Budget in millions USD")


model_with_structure=model.with_structured_output(MovieDetails)

response=model_with_structure.invoke("Provide the details of the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Dileep Rao', role='Yusuf'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Tom Berenger', role='Prophet'), Actor(name='Pete Postlethwaite', role='Maurice Fischer')], genres=['Action', 'Thriller', 'Sci-Fi', 'Mystery'], budget=160.0)

In [56]:
response=model_with_structure.invoke("Provide the details of the Doctor strange movie")
response

MovieDetails(title='Doctor Strange', year=2016, cast=[Actor(name='Benedict Cumberbatch', role='Doctor Stephen Strange')], genres=['Fantasy', 'Action', 'Adventure'], budget=165.0)

## Step 7: Structuring output using TypedDict
You can also use Python's built-in `TypedDict` to get parsed results as native Python dictionaries instead of Pydantic objects.

In [66]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A Movie with the details."""
    title:Annotated[str,...,"The title of the movie"]
    year:Annotated[int,...,"Year in which the movie was released"]
    director:Annotated[str,...,"Director of the movie"]
    rating:Annotated[float,...,"Movie rating out of 10"]

model_with_dict=model.with_structured_output(MovieDict)
response=model_with_dict.invoke("Provide the details of the movie Geetha Govindam.")
response

{'director': 'Parasuram',
 'rating': 7.5,
 'title': 'Geetha Govindam',
 'year': 2018}

### Step 7.1: Nested TypedDict
We can use nesting inside TypedDict structures as well.

In [69]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None =Field(None,description="Budget in millions USD")


model_with_structure=model.with_structured_output(MovieDetails)

response=model_with_structure.invoke("Provide the details of the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark/Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers/Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner/Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff/Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Albert Russo/Hawkeye'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'}],
 'genres': ['Action', 'Adventure', 'Science Fiction'],
 'title': 'Avengers',
 'year': 2012}

## Step 8: View Model Profile Capabilities

In [70]:
model.profile

{'name': 'Llama 3.1 8B Instant',
 'release_date': '2024-07-23',
 'last_updated': '2024-07-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 131072,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

## Step 9: Structured Outputs in LangChain Agents (`response_format`)
In LangChain, we can direct agents to reply in a structured output format by passing our schema to the `response_format` parameter in the `create_agent` call.

In [77]:
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name:str=Field(description="Name of the person")
    email:str=Field(description="Email address of the person")
    phone:int=Field(description="Phone number of the person")

agent=create_agent(
    model=model,
    response_format=ContactInfo
) 

result = agent.invoke({
    "messages":[{"role":"user","content":"Extarct contact info from : Tejeswar, Acharytejeswar@gmail.com,9490217446"}]
})
print(result["structured_response"])

name='Tejeswar' email='acharytejeswar@gmail.com' phone=9490217446


### Step 9.1: Extracting Financial Transactions
We use the same structured agent concept to extract financial transactions from unstructured user transcripts.

In [82]:
from google.genai.types import File
class TransactionInfo(BaseModel):
    """Transaction information of a person."""
    category:str=Field(description="category like food, transport, entertainment")
    cashflow:str=Field(description="Credit or debit of the amount.")
    amount:float=Field(description="Amount transactioned form the bank")
    date:str=Field(description="Date information of the transaction")
    details:str=Field(description="details of the amount used for or if credits source of credit.")

agent=create_agent(
    model=model,
    response_format=TransactionInfo
) 
result=agent.invoke({
    "messages":[{"role":"user","content":"Extarct transaction info from : Hey, I just checked my banking app. On May 14, 2026, I spent $45.50 at Starbucks for coffee and pastries with the team. It shows up as a deduction on my account."
}]
}

)
print(result["structured_response"])

category='food' cashflow='debit' amount=45.5 date='May 14, 2026' details='Coffee and pastries with the team at Starbucks'


In [85]:
result=agent.invoke({
    "messages":[{"role":"user","content":"Extarct transaction info from : Venmo Payment: You sent $120.75 to your roommate John on 04/12/2026 for your share of the electric and wifi bills."
        }]}

)
print(result["structured_response"])

category='bills' cashflow='debit' amount=120.75 date='04/12/2026' details='share of electric and wifi bills'


### Step 9.2: Agent Structured Output with TypedDict

In [ ]:
#using type dict
class ContactInfodict(TypedDict):
    """Contact information of a person."""
    name:str
    email:str
    phone:str

agent=create_agent(
    model=model,
    response_format=ContactInfodict
) 
result=agent.invoke({
    "messages":[{"role":"user","content":"Extarct contact info from : Tejeswar, acharytejeswar@gmail.com,9490217446"
}]
}

)
print(result["structured_response"])


{'name': 'Tejeswar', 'email': 'acharytejeswar@gmail.com', 'phone': '9490217446'}


## Step 10: Agent Structured Output using Python Dataclasses

In [98]:
from dataclasses import dataclass

@dataclass
class ContactInfo:
    """Contact information of a person."""
    name:str
    email:str
    phone:str


agent=create_agent(
    model=model,
    response_format=ContactInfo
) 
result=agent.invoke({
    "messages":[{"role":"user","content":"Extarct contact info from : Tejeswar, acharytejeswar@gmail.com,9490217446"
}]
}

)
result["structured_response"]

    

ContactInfo(name='Tejeswar', email='acharytejeswar@gmail.com', phone='9490217446')